# Qualitative TTC — curated scenes

Scenes and splits default to **v1.0-trainval** and the full **val** set (set `NUSCENES_VER` / `NUSCENES_SPLIT` to use the mini devkit if needed). Below is the methodology to find and analyze different scenes. The steps I use are below:

1. Use descriptions that nuScenes are annotated with to find different types of scenes that might be interesting (e.g., at an intersection, straight-away, parking lot, etc.)
2. Run **`compare_ttc_scene`** per row → plots + optional MP4 (GT / physics / MLP).
4. Run **`visualize_ttc_bev`** per row → GT TTC–colored BEV

### Use this code block to find specific tokens of interest you might want to analyze

In [2]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

NUSCENES_ROOT = os.environ.get("NUSCENES_ROOT", str(Path("..").resolve() / "data" / "nuscenes"))
NUSCENES_VER = os.environ.get("NUSCENES_VER", "v1.0-trainval")
SPLIT = os.environ.get(
    "NUSCENES_SPLIT",
    "mini_val" if NUSCENES_VER == "v1.0-mini" else "val",
)

from nuscenes.nuscenes import NuScenes
from nuscenes.utils.splits import create_splits_scenes

nusc = NuScenes(version=NUSCENES_VER, dataroot=NUSCENES_ROOT, verbose=False)

# --- filter scenes by description (case-insensitive) ---
# Optional: only among a split (set SPLIT=None to search all scenes in this DB)
SPLIT = "val"  # or "val", None, ...

from nuscenes.utils.splits import create_splits_scenes

def scenes_to_search(nusc, split):
    rows = list(nusc.scene)
    if not split:
        return rows
    names = set(create_splits_scenes()[split])
    return [s for s in rows if s["name"] in names]

candidates = scenes_to_search(nusc, SPLIT)

# Match if ANY keyword appears in description
KEYWORDS = ["intersection", "turn"]  # e.g. change to ["parking"], ["highway"], ["pedestrian"], etc.
kw = [k.lower() for k in KEYWORDS]

def match_desc(scene_rec):
    d = (scene_rec.get("description") or "").lower()
    return any(k in d for k in kw)

matched = [s for s in candidates if match_desc(s)]
print(f"Matched {len(matched)} scene(s) with KEYWORDS={KEYWORDS} (split={SPLIT!r})\n")
for s in sorted(matched, key=lambda x: x["name"]):
    print(f"{s['name']:16}  {s['token']}")
    print(f"  {s.get('description', '')}\n")

Matched 79 scene(s) with KEYWORDS=['intersection', 'turn'] (split='val')

scene-0012        265f002f02d447ad9074813292eef75e
  Nature, bus stop, parked cars, bus exits from the intersection

scene-0014        d1e57234fd6a463d963670938f9f556e
  Follow bus, cross intersection, nature

scene-0016        efa5c96f05594f41a2498eb9f2e7ad99
  Turn left, people on both sides, parked vans

scene-0017        3dd9ad3f963e4f588d75c112cbf07f56
  Barriers, peds, arriving at busy intersection

scene-0018        b51869782c0e464b8021eb798609f35f
  Merge left, turn into unloading zone

scene-0093        cba3ddd5c3664a43b6a08e586e094900
  Turn right, parking lots, cross intersection

scene-0094        91c071bcc1ad4fa1b555399e1cfbab79
  Oncoming cyclist, cars, wait at intersection

scene-0095        b4b82c4d338a4b6d86835388ce076345
  Wait at intersection, peds occluded, start driving

scene-0096        68e79a88244f447f993a72da444b29ba
  Cross busy intersection, cyclist nearby, construction barriers and con

In [1]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC = REPO_ROOT / "src"
EV_CMP = SRC / "tools" / "compare_ttc_scene.py"
EV_BEV = SRC / "tools" / "visualize_ttc_bev.py"
QUAL_OUT = REPO_ROOT / "work_dirs" / "qualitative_ttc"
QUAL_OUT.mkdir(parents=True, exist_ok=True)

NUSCENES_ROOT = os.environ.get("NUSCENES_ROOT", str(REPO_ROOT / "data" / "nuscenes")).rstrip("/") + "/"
NUSCENES_VER = os.environ.get("NUSCENES_VER", "v1.0-trainval")
ANN_FILE = os.environ.get("ANN_FILE", str(REPO_ROOT / "data" / "nuscenes" / "nuscenes2d_temporal_infos_val.pkl"))
TTC_PKL = os.environ.get("STREAMPETR_TTC_PKL", str(REPO_ROOT / "data" / "nuscenes" / "ttc_gt_labels_v1_0_trainval.pkl"))

CONFIG = str(SRC / "projects" / "configs" / "StreamPETR_ttc_v3" / "stream_petr_vov_ttc_frozen_20e.py")
CHECKPOINT = os.environ.get("CHECKPOINT", str(REPO_ROOT / "work_dirs" / "streampetr_ttc_v3_frozen_20e" / "latest.pth"))
PRETRAINED_BASELINE = os.environ.get("PRETRAINED_BASELINE", str(REPO_ROOT / "ckpts" / "stream_petr_vov_flash_800_bs2_seq_24e.pth"))

GPU_ID = int(os.environ.get("GPU_ID", "0"))
MAX_CAM_PANELS = 6
RUN_COMPARE = os.environ.get("RUN_COMPARE", "1") == "1"
RUN_BEV = os.environ.get("RUN_BEV", "1") == "1"
RUN_VIDEO = os.environ.get("RUN_VIDEO", "0") == "1"
VIDEO_PANELS = os.environ.get("VIDEO_PANELS", "gt,physics,mlp")

_env = {**os.environ, "PYTHONPATH": str(SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")}

def _run(cmd):
    print("::", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=str(REPO_ROOT), env=_env, check=False)
    return r.returncode

# One row per qualitative regime; paste scene_token from nuScenes (see next cell for val list)
SCENES = pd.DataFrame(
    [
        {"regime": "intersection", "scene_token": "c525507ee2ef4c6d8bb64b0e0cf0dd32", "why_picked": "fast cross traffic"},
        {"regime": "straight-away", "scene_token": "40209c4e465d4b4e8341ebd52be0d842", "why_picked": "same-direction, slow closing"},
        {"regime": "parking-lot", "scene_token": "e60ef590e3614187b7800db3e5284e1a", "why_picked": "near-static, low closing"},
        {"regime": "near-miss", "scene_token": "68e79a88244f447f993a72da444b29ba", "why_picked": "short GT TTC"},
    ]
)
SCENES


,regime,scene_token,why_picked
0,intersection,c525507ee2ef4c6d8bb64b0e0cf0dd32,fast cross traffic
1,straight-away,40209c4e465d4b4e8341ebd52be0d842,"same-direction, slow closing"
2,parking-lot,e60ef590e3614187b7800db3e5284e1a,"near-static, low closing"
3,near-miss,68e79a88244f447f993a72da444b29ba,short GT TTC


### Running comparisons for these scenes
Ran comparisons for the scenes above in a slurm script with GPU attached so would run faster thus not in this notebook. Results are detailed below.
